# Comparing Routes

Now let's run Yen's and compare algorithmic recommendations to actual historical operations.

This comparison reveals where the biggest optimization opportunities exist.

## Setup: Connect to AGA

In [ ]:
import os
import pandas as pd
from datetime import timedelta
from graphdatascience.session import GdsSessions, AuraAPICredentials
from graphdatascience.session import DbmsConnectionInfo, SessionMemory
from dotenv import load_dotenv

pd.set_option('display.max_colwidth', None)
pd.set_option('display.width', None)

# Import neo4j driver Result for graph transformations
from neo4j import Result

# Import neo4j_viz for visualization
from neo4j_viz.gds import from_gds
from neo4j_viz.neo4j import from_neo4j

# Load environment variables
load_dotenv()

# Get Aura API credentials
client_id = os.getenv('AURA_CLIENT_ID')
client_secret = os.getenv('AURA_CLIENT_SECRET')
project_id = os.getenv('AURA_PROJECT_ID')  # set in .env only if your Aura account has multiple projects

# Get AuraDB connection info
uri = os.getenv('AURA_URI')
username = os.getenv('AURA_USERNAME')
password = os.getenv('AURA_PASSWORD')

# Create sessions manager
sessions = GdsSessions(
    api_credentials=AuraAPICredentials(client_id, client_secret, project_id=project_id)
)

print("Sessions manager created")

In [ ]:
# Create a GDS Session
gds = sessions.get_or_create(
    session_name="comparing-routes",
    memory=SessionMemory.m_2GB,
    db_connection=DbmsConnectionInfo(
        uri=uri,
        username=username,
        password=password
    ),
    ttl=timedelta(minutes=30)
)

gds.verify_connectivity()
print(f"Connected to GDS Session: comparing-routes")

## Step 1: Create the Projection

Let's project the logistics network with aggregated transit times.

### Native projection (new in Aura Graph Analytics)

Native projections read node labels and relationship types directly from database storage - no Cypher query - the fastest way to bulk-load a graph into a session.

Note: native projections load raw relationships and cannot aggregate. This logistics graph has up to ~1,500 parallel transit edges per route, which we need to average. So here native is a feature demo; the Cypher projection below (which aggregates effectiveMinutes) is what Yen's uses.

In [ ]:
# Native projection - reads labels + relationship types straight from storage (fastest bulk load).
# Demo only: loads RAW (un-aggregated) edges, so we don't use it for Yen's below.
G_native, _ = gds.v2.graph.project_native(
    "logistics-network-native",
    ["EntryPoint", "DeparturePoint", "DepartureWarehouse",
     "TransferPoint", "ArrivalWarehouse", "Destination"],
    ["RECEPTION", "DEPARTURE", "TRANSPORT", "DELIVERY"],
    relationship_properties=["effectiveMinutes"],
)
print(f"Native projection: {G_native.node_count():,} nodes, "
      f"{G_native.relationship_count():,} relationships (raw, un-aggregated)")
G_native.drop()  # demo only - the analysis below uses the aggregated Cypher projection

In [ ]:
# Project the logistics network
G, result = gds.graph.project(
    "logistics-network-cypher",
    """
    CALL {
        MATCH (source)
        WHERE source:EntryPoint OR source:DeparturePoint OR source:DepartureWarehouse
           OR source:TransferPoint OR source:ArrivalWarehouse OR source:Destination
        OPTIONAL MATCH (source)-[r:RECEPTION|DEPARTURE|TRANSPORT|DELIVERY]->(target)
        WITH source, target, type(r) AS relType,
             avg(r.effectiveMinutes) AS avgMinutes
        RETURN source, target, relType, avgMinutes
    }
    RETURN gds.graph.project.remote(source, target, {
        sourceNodeLabels: labels(source),
        targetNodeLabels: labels(target),
        relationshipType: relType,
        relationshipProperties: {avgMinutes: avgMinutes}
    })
    """
)

print(f"Projected graph: {G.name()}")
print(f"  Nodes: {G.node_count():,}")
print(f"  Relationships: {G.relationship_count():,}")

## Step 2: Run Yen's K-Shortest Paths

Find the top 10 routes from Moodytown to Davisfort.

In [ ]:
# Get source and target node IDs
source_id = gds.find_node_id(["EntryPoint"], {"name": "Moodytown"})
target_id = gds.find_node_id(["Destination"], {"name": "Davisfort"})

print(f"Source: Moodytown (ID: {source_id})")
print(f"Target: Davisfort (ID: {target_id})")

In [ ]:
# Run Yen's algorithm for top 10 paths
yens_result = gds.shortestPath.yens.stream(
    G,
    sourceNode=source_id,
    targetNode=target_id,
    k=10,
    relationshipWeightProperty='avgMinutes'
)

print(yens_result)

Let's take a look at that in a more readable format:

In [ ]:
# Process and display results
algorithm_routes = []
for idx, row in yens_result.iterrows():
    # Get location names along the path
    path_names = gds.run_cypher("""
        UNWIND $nodeIds AS nodeId
        RETURN gds.util.asNode(nodeId).name AS name
    """, params={"nodeIds": list(row['nodeIds'])})
    
    # Remove duplicate consecutive names
    airports = []
    prev_name = None
    for name in path_names['name'].tolist():
        if name != prev_name:
            airports.append(name)
            prev_name = name
    
    algorithm_routes.append({
        'rank': row['index'] + 1,
        'route': ' → '.join(airports),
        'minutes': round(row['totalCost']),
        'days': round(row['totalCost'] / 1440, 2),
        'node_ids': list(row['nodeIds'])
    })

algorithm_df = pd.DataFrame(algorithm_routes)
print("Top 10 Routes (Yen's Algorithm):")
print(algorithm_df[['rank', 'route', 'minutes', 'days']].to_string(index=False))

## Understanding the Results

You now have 10 ranked routes. Each route is progressively longer but still viable.

| Rank | Airports | Minutes | Days |
|------|----------|---------|------|
| 1 | Moodytown → Davisfort | 3743 | 2.6 |
| 2 | Moodytown → Michaelstad → Davisfort | 4410 | 3.06 |
| 3 | Moodytown → Wandaborough → Davisfort | 4589 | 3.19 |
| ... | ... | ... | ... |

### Visualize: Top 10 Routes

In [ ]:
# Helper function: Execute query with Neo4j driver (for visualizations)
def execute_neo4j_query(query, params=None, database=os.getenv("AURA_DATABASE") or "neo4j"):
    from neo4j import GraphDatabase, RoutingControl
    with GraphDatabase.driver(uri, auth=(username, password)) as driver:
        driver.verify_connectivity()
        result = driver.execute_query(
            query,
            parameters_=params or {},
            database_=database,
            routing_=RoutingControl.READ,
            result_transformer_=Result.graph,
        )
    return result

# Helper function: Create visualization from Cypher query
def visualize_query(query, params=None, database=os.getenv("AURA_DATABASE") or "neo4j"):
    result = execute_neo4j_query(query, params, database)
    return from_neo4j(result)

# Helper function: Visualize a GDS graph projection
def visualize_projection(G):
    return from_gds(gds, G, max_node_count=50)

# Helper function: Visualize routes from Yen's result
def visualize_routes(yens_result, title="Routes"):
    """Visualize all routes from Yen's result."""
    # Collect all unique node IDs from all paths
    all_node_ids = set()
    for idx, row in yens_result.iterrows():
        all_node_ids.update(row['nodeIds'])
    
    # Get location names
    names_df = gds.run_cypher("""
        UNWIND $nodeIds AS nodeId
        RETURN gds.util.asNode(nodeId).name AS name
    """, params={"nodeIds": list(all_node_ids)})
    names = list(set(names_df['name'].tolist()))
    
    # Visualize paths between these locations
    print(title)
    VG = visualize_query(f"""
        MATCH path = SHORTEST 1 (a)-[r:RECEPTION|DEPARTURE|TRANSPORT|DELIVERY]->(b)
        WHERE a.name IN {names} AND b.name IN {names}
        RETURN path
    """)
    return VG

print("Helper functions loaded")

In [ ]:
# Visualize all top 10 routes
VG = visualize_routes(yens_result, "Top 10 Routes: Moodytown → Davisfort")
VG.render()

## Step 3: Compare to Historical Routes

Now, let's take a look at what routes were actually taken historically.

First let's get the airport ids:

In [ ]:
# Get historical routes
historical_routes = gds.run_cypher("""
    MATCH (source:EntryPoint {name: $sourceName})
      -[:HAS_HISTORICAL_ROUTE]->(hr:HistoricalRoute)
      -[:TERMINATES_AT]->(target:Destination {name: $targetName})
    RETURN hr.airportPath AS route,
           hr.pathCount AS shipment_count,
           hr.avgCostMin AS avg_minutes
    ORDER BY hr.pathCount DESC
""", params={"sourceName": "Moodytown", "targetName": "Davisfort"})

print("Historical Routes (Most Used):")
print(historical_routes.to_string(index=False))

## The Comparison

Let's compare the historical routes with the algorithmic recommendations side by side.

In [ ]:
# Display comparison
print("=" * 70)
print("HISTORICAL ROUTES (Most Used)")
print("=" * 70)
for idx, row in historical_routes.iterrows():
    print(f"  {row['route']}")
    print(f"    Count: {row['shipment_count']} shipments | Avg: {row['avg_minutes']:,.0f} minutes")
    print()

print("=" * 70)
print("ALGORITHM RECOMMENDATIONS (Top 3)")
print("=" * 70)
for idx, row in algorithm_df.head(3).iterrows():
    print(f"  Rank {row['rank']}: {row['route']}")
    print(f"    Time: {row['minutes']:,} minutes ({row['days']} days)")
    print()

## How to Interpret the Results

We can see here that, historically, we have not been utilising the optimal routes. For example, Scotttown has been used 11 times -- and doesn't appear in even the top ten.

Wandaborough, as the third-most optimal route has only ever been used once. 

## Quantifying the Opportunity

Let's calculate the total potential savings from optimizing suboptimal historical routes.

In [ ]:
# Calculate optimization opportunity
best_algo_time = algorithm_df.iloc[0]['minutes']
third_best_time = algorithm_df.iloc[2]['minutes'] if len(algorithm_df) > 2 else best_algo_time

print("Optimization Opportunity Analysis")
print("=" * 70)

total_actual_time = 0
total_optimal_time = 0
total_shipments = 0

for idx, row in historical_routes.iterrows():
    shipments = row['shipment_count']
    actual_time = row['avg_minutes'] * shipments
    optimal_time = third_best_time * shipments  # Use 3rd best as conservative estimate
    
    total_actual_time += actual_time
    total_optimal_time += optimal_time
    total_shipments += shipments
    
    if row['avg_minutes'] > third_best_time:
        savings = actual_time - optimal_time
        print(f"\n{row['route']}")
        print(f"  Shipments: {shipments}")
        print(f"  Actual time:  {shipments} × {row['avg_minutes']:,.0f} = {actual_time:,.0f} minutes")
        print(f"  Optimal time: {shipments} × {third_best_time:,.0f} = {optimal_time:,.0f} minutes")
        print(f"  Wasted time:  {savings:,.0f} minutes ({savings/1440:.1f} days)")

total_wasted = total_actual_time - total_optimal_time
print("\n" + "=" * 70)
print(f"TOTAL OPTIMIZATION OPPORTUNITY")
print(f"  Total shipments analyzed: {total_shipments}")
print(f"  Total actual time:  {total_actual_time:,.0f} minutes ({total_actual_time/1440:.1f} days)")
print(f"  Total optimal time: {total_optimal_time:,.0f} minutes ({total_optimal_time/1440:.1f} days)")
print(f"  Total potential savings: {total_wasted:,.0f} minutes ({total_wasted/1440:.1f} days)")
if total_actual_time > 0:
    print(f"  Improvement: {(total_wasted/total_actual_time)*100:.1f}%")

With this information, we could now present a report, quantifying the opportunity to leadership. 

## Step 4: Find More Optimization Targets

Let's see if we can identify which historical routes across the entire network need review.

We'll look for high-traffic routes with high transit times. 

In [ ]:
# Find high-volume routes with long transit times
optimization_targets = gds.run_cypher("""
    MATCH (source:EntryPoint)-[:HAS_HISTORICAL_ROUTE]->(hr:HistoricalRoute)-[:TERMINATES_AT]->(target:Destination)
    WHERE hr.pathCount > 5
    RETURN source.name AS origin,
           target.name AS destination,
           hr.airportPath AS current_route,
           hr.avgCostMin AS avg_minutes,
           hr.pathCount AS shipment_count,
           round(hr.avgCostMin / 1440.0, 2) AS avg_days
    ORDER BY hr.avgCostMin DESC
    LIMIT 10
""")

print("Top 10 Optimization Targets (High-Volume, Long Transit Time):")
print(optimization_targets.to_string(index=False))

## Cleanup

In [ ]:
# Drop the projection
G.drop()

In [ ]:
# Delete the session
gds.delete()
print("Session deleted - billing stopped")

## Summary

You've compared algorithmic recommendations to historical operations:

* **Found 10 ranked alternative routes** with Yen's
* **Identified suboptimal routes** that aren't in the top 10
* **Quantified potential savings** (often 40%+ improvement possible)
* **Found more optimization targets** across the network

The power of pathfinding isn't just finding routes—it's **revealing the gap between optimal and actual operations**.